# Bradford Bulls — Body Part Visibility Analysis

Maps every player pixel in the video to one of **24 body regions** (head, torso, arms, legs, feet)
using **DensePose**, then aggregates across all sampled frames to answer:
*which body parts are most visible — and therefore most valuable for sponsor placement?*

**Pipeline:**
1. Sample frames from video at a configurable FPS
2. Run DensePose → 24-region coloured body map per frame
3. Count pixels per body part across all persons and frames
4. Visualise: sample overlays + grouped bar chart + CSV export

**Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — switch to T4 GPU"}')

## 2. Install Detectron2 + DensePose

In [ ]:
# First run only — takes ~5–8 min
# Clone full repo so config YAML files are available locally
!git clone --depth=1 https://github.com/facebookresearch/detectron2 /tmp/detectron2_repo 2>/dev/null \
    || echo 'Repo already cloned'

# Install packages from local clone
!pip install -q /tmp/detectron2_repo
!pip install -q /tmp/detectron2_repo/projects/DensePose
print('Installation complete — restart runtime if prompted')

## 3. Mount Drive + Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# >>> UPDATE path to your video <<<
VIDEO_PATH = '/content/drive/MyDrive/bradford_bulls/videos/M02_h264.mp4'

# ── Sampling ──────────────────────────────────────────────────────────────────
SAMPLE_FPS  = 1.0   # frames per second to sample (1 = 1 frame/sec)
MAX_FRAMES  = 500   # cap total frames analysed (set None for no cap)

# ── Detection ─────────────────────────────────────────────────────────────────
CONF_THRESH = 0.70  # minimum person detection confidence

# ── Output ────────────────────────────────────────────────────────────────────
stem       = Path(VIDEO_PATH).stem
OUTPUT_DIR = Path(f'/content/drive/MyDrive/bradford_bulls/bodypart_analysis/{stem}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Video  : {VIDEO_PATH}')
print(f'Exists : {Path(VIDEO_PATH).exists()}')
print(f'Output : {OUTPUT_DIR}')

## 4. Load DensePose Model

In [ ]:
import os, torch
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from densepose import add_densepose_config

# ── Config path — from cloned repo (reliable) ────────────────────────────────
CONFIG_PATH = (
    '/tmp/detectron2_repo/projects/DensePose/configs/'
    'densepose_rcnn_R_50_FPN_s1x.yaml'
)
if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(
        f'Config not found at {CONFIG_PATH}.\n'
        'Re-run the install cell (make sure the git clone succeeded), '
        'then restart the runtime.'
    )
print(f'Config : {CONFIG_PATH}')

# ── Build predictor ───────────────────────────────────────────────────────────
cfg = get_cfg()
add_densepose_config(cfg)
cfg.merge_from_file(CONFIG_PATH)
cfg.MODEL.WEIGHTS = (
    'https://dl.fbaipublicfiles.com/densepose/'
    'densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl'
)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = CONF_THRESH
cfg.MODEL.DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

predictor = DefaultPredictor(cfg)
print(f'DensePose ready on {cfg.MODEL.DEVICE.upper()}')

## 5. Body Part Colours + Groups

In [ ]:
import numpy as np

# ── 24 DensePose part names ───────────────────────────────────────────────────
PART_NAMES = {
     0: 'Background',
     1: 'Torso Front',   2: 'Torso Back',
     3: 'Right Hand',    4: 'Left Hand',
     5: 'Left Foot',     6: 'Right Foot',
     7: 'Upper Leg R',   8: 'Upper Leg L',
     9: 'Upper Leg R',  10: 'Upper Leg L',
    11: 'Lower Leg R',  12: 'Lower Leg L',
    13: 'Lower Leg R',  14: 'Lower Leg L',
    15: 'Upper Arm L',  16: 'Upper Arm R',
    17: 'Upper Arm L',  18: 'Upper Arm R',
    19: 'Lower Arm L',  20: 'Lower Arm R',
    21: 'Lower Arm L',  22: 'Lower Arm R',
    23: 'Head',         24: 'Head',
}

# ── RGB colours — close to DensePose paper palette ────────────────────────────
PART_COLORS = np.array([
    [  0,   0,   0],  #  0  background
    [255, 140,   0],  #  1  Torso Front    orange
    [255, 180,  80],  #  2  Torso Back     light orange
    [139,   0,   0],  #  3  Right Hand     dark red
    [220,  20,  60],  #  4  Left Hand      crimson
    [255, 165,   0],  #  5  Left Foot      orange
    [255, 215,   0],  #  6  Right Foot     gold
    [  0, 180,   0],  #  7  Upper Leg R F  green
    [144, 238, 144],  #  8  Upper Leg L F  light green
    [  0, 100,   0],  #  9  Upper Leg R B  dark green
    [  0,  50,   0],  # 10  Upper Leg L B  very dark green
    [255, 255,   0],  # 11  Lower Leg R F  yellow
    [173, 255,  47],  # 12  Lower Leg L F  green-yellow
    [128, 128,   0],  # 13  Lower Leg R B  olive
    [ 85, 107,  47],  # 14  Lower Leg L B  dark olive
    [  0,   0, 255],  # 15  Upper Arm L F  blue
    [ 30, 144, 255],  # 16  Upper Arm R F  dodger blue
    [  0,   0, 139],  # 17  Upper Arm L B  dark blue
    [  0,   0,  80],  # 18  Upper Arm R B  navy
    [255,   0, 255],  # 19  Lower Arm L F  magenta
    [218, 112, 214],  # 20  Lower Arm R F  orchid
    [139,   0, 139],  # 21  Lower Arm L B  dark magenta
    [ 75,   0, 130],  # 22  Lower Arm R B  indigo
    [  0, 206, 209],  # 23  Head Right     dark turquoise
    [135, 206, 235],  # 24  Head Left      sky blue
], dtype=np.uint8)

# ── Grouped parts for summary chart ──────────────────────────────────────────
GROUPS = {
    'Head':        {'ids': [23, 24],      'color': '#00BFFF'},
    'Torso':       {'ids': [1,  2],       'color': '#FF8C00'},
    'Upper Arm L': {'ids': [15, 17],      'color': '#0000FF'},
    'Upper Arm R': {'ids': [16, 18],      'color': '#1E90FF'},
    'Lower Arm L': {'ids': [19, 21],      'color': '#FF00FF'},
    'Lower Arm R': {'ids': [20, 22],      'color': '#DA70D6'},
    'Hand L':      {'ids': [4],           'color': '#DC143C'},
    'Hand R':      {'ids': [3],           'color': '#8B0000'},
    'Upper Leg L': {'ids': [8,  10],      'color': '#90EE90'},
    'Upper Leg R': {'ids': [7,  9],       'color': '#008000'},
    'Lower Leg L': {'ids': [12, 14],      'color': '#ADFF2F'},
    'Lower Leg R': {'ids': [11, 13],      'color': '#FFFF00'},
    'Foot L':      {'ids': [5],           'color': '#FFA500'},
    'Foot R':      {'ids': [6],           'color': '#FFD700'},
}
print(f'{len(GROUPS)} body part groups defined')

## 6. Sample Frames

In [ ]:
import cv2

cap        = cv2.VideoCapture(VIDEO_PATH)
src_fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_f    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_step = max(1, int(round(src_fps / SAMPLE_FPS)))
cap.release()

frame_indices = list(range(0, total_f, frame_step))
if MAX_FRAMES and len(frame_indices) > MAX_FRAMES:
    step2        = max(1, len(frame_indices) // MAX_FRAMES)
    frame_indices = frame_indices[::step2][:MAX_FRAMES]

print(f'Video    : {src_fps:.1f} fps, {total_f} frames ({total_f/src_fps/60:.1f} min)')
print(f'Sampling : every {frame_step} frames → {len(frame_indices)} frames to process')

## 7. Run DensePose Inference

In [ ]:
import torch
from tqdm.notebook import tqdm


def run_densepose(frame_bgr):
    """Run DensePose on one frame. Returns (vis_rgb, part_counts[25])."""
    H, W      = frame_bgr.shape[:2]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    counts    = np.zeros(25, dtype=np.int64)

    with torch.no_grad():
        outputs = predictor(frame_bgr)

    instances = outputs['instances'].to('cpu')
    if not instances.has('pred_densepose') or len(instances) == 0:
        return frame_rgb, counts

    dp          = instances.pred_densepose
    boxes       = instances.pred_boxes.tensor.numpy().astype(int)
    part_labels = dp.fine_segm.argmax(dim=1).numpy()    # (N, H_roi, W_roi)
    fg_masks    = dp.coarse_segm.argmax(dim=1).numpy()  # (N, H_roi, W_roi) 1=person 0=bg

    overlay = frame_rgb.copy()

    for lmap, fg, box in zip(part_labels, fg_masks, boxes):
        x1 = max(0, box[0]); y1 = max(0, box[1])
        x2 = min(W, box[2]); y2 = min(H, box[3])
        if x2 <= x1 or y2 <= y1:
            continue

        bw, bh = x2 - x1, y2 - y1
        lmap_r = cv2.resize(lmap.astype(np.uint8), (bw, bh), interpolation=cv2.INTER_NEAREST)
        fg_r   = cv2.resize(fg.astype(np.uint8),   (bw, bh), interpolation=cv2.INTER_NEAREST)

        for pid in range(1, 25):
            # Only colour pixels that are foreground AND belong to this part
            mask = (lmap_r == pid) & (fg_r == 1)
            if not mask.any():
                continue
            counts[pid] += int(mask.sum())
            overlay[y1:y2, x1:x2][mask] = PART_COLORS[pid]

    vis = (0.65 * overlay + 0.35 * frame_rgb).astype(np.uint8)
    return vis, counts


# ── Main loop ─────────────────────────────────────────────────────────────────
total_counts = np.zeros(25, dtype=np.int64)
sample_vis   = []
SAVE_EVERY   = max(1, len(frame_indices) // 6)

cap = cv2.VideoCapture(VIDEO_PATH)
for i, fidx in enumerate(tqdm(frame_indices, desc='DensePose')):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ok, frame = cap.read()
    if not ok:
        continue
    vis, counts = run_densepose(frame)
    total_counts += counts
    if i % SAVE_EVERY == 0:
        sample_vis.append(vis)
cap.release()

total_person_px = int(total_counts[1:].sum())
print(f'Processed   : {len(frame_indices)} frames')
print(f'Person px   : {total_person_px:,}')
print(f'Preview vis : {len(sample_vis)} frames')

## 8. Visualise Sample Frames (DensePose overlay)

In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

n    = len(sample_vis)
cols = min(3, n)
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 3.5))
axes = np.array(axes).flatten()

for i, vis in enumerate(sample_vis):
    axes[i].imshow(vis)
    axes[i].set_title(f'frame {frame_indices[i * SAVE_EVERY]:,}', fontsize=9)
    axes[i].axis('off')
for j in range(n, len(axes)):
    axes[j].axis('off')

# Colour legend (24 parts)
patches = [
    mpatches.Patch(color=PART_COLORS[pid] / 255, label=f'{PART_NAMES[pid]} ({pid})')
    for pid in range(1, 25)
]
fig.legend(handles=patches, loc='lower center', ncol=6,
           fontsize=7, bbox_to_anchor=(0.5, -0.02))

plt.suptitle(f'DensePose overlay — {stem}', fontsize=13, fontweight='bold')
plt.tight_layout()
out = OUTPUT_DIR / 'sample_overlays.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

## 9. Aggregate Results + Bar Chart

In [ ]:
import pandas as pd

if total_person_px == 0:
    print('No person pixels detected — lower CONF_THRESH or check VIDEO_PATH.')
else:
    rows_data = []
    for name, info in GROUPS.items():
        px = sum(int(total_counts[pid]) for pid in info['ids'])
        rows_data.append({
            'body_part': name,
            'pixels':    px,
            'pct':       round(100 * px / total_person_px, 2),
            'color':     info['color'],
        })

    df = (pd.DataFrame(rows_data)
            .sort_values('pct', ascending=False)
            .reset_index(drop=True))
    df.insert(0, 'rank', df.index + 1)

    print(df[['rank', 'body_part', 'pixels', 'pct']].to_string(index=False))

    # ── Horizontal bar chart ──────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 7))
    bars = ax.barh(
        df['body_part'][::-1], df['pct'][::-1],
        color=df['color'][::-1].tolist(),
        edgecolor='white', linewidth=0.5
    )
    for bar, pct in zip(bars, df['pct'][::-1]):
        ax.text(bar.get_width() + 0.3,
                bar.get_y() + bar.get_height() / 2,
                f'{pct:.1f}%', va='center', fontsize=9)

    ax.set_xlabel('% of total person pixels', fontsize=11)
    ax.set_title(
        f'Body Part Visibility — {stem}\n'
        f'({len(frame_indices)} frames sampled @ {SAMPLE_FPS} fps)',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlim(0, df['pct'].max() * 1.18)
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    chart_path = OUTPUT_DIR / 'bodypart_visibility.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Chart saved → {chart_path}')

## 10. Export CSV

In [ ]:
csv_path = OUTPUT_DIR / f'{stem}_bodypart_visibility.csv'
df.drop(columns=['color']).to_csv(csv_path, index=False)
print(f'CSV saved → {csv_path}')
print(df[['rank', 'body_part', 'pct']].to_string(index=False))

## 10. Generate DensePose Video

Process every frame of the clip and write a new MP4 with the coloured body-part overlay baked in.\
`VIDEO_PROC_STEP = 1` → every frame (best quality); `2` → every other frame (2× faster, acceptable for short clips.

In [ ]:
import subprocess
from tqdm.notebook import tqdm

VIDEO_PROC_STEP = 1     # 1=every frame, 2=every other frame (2x faster)
OVERLAY_ALPHA   = 0.65  # 0=no overlay … 1=full colour

cap      = cv2.VideoCapture(VIDEO_PATH)
src_fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_f  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'Source : {W}x{H} @ {src_fps:.1f} fps — {total_f} frames ({total_f/src_fps:.1f} s)')
print(f'Step   : every {VIDEO_PROC_STEP} frame(s)')
print(f'Est.   : ~{total_f / VIDEO_PROC_STEP / 7:.0f} s on T4\n')

TMP_PATH = '/tmp/dp_overlay_raw.mp4'
writer   = cv2.VideoWriter(TMP_PATH, cv2.VideoWriter_fourcc(*'mp4v'), src_fps, (W, H))

last_vis_bgr = None

for fidx in tqdm(range(total_f), desc='DensePose video'):
    ok, frame = cap.read()
    if not ok:
        break

    if fidx % VIDEO_PROC_STEP == 0:
        with torch.no_grad():
            outputs = predictor(frame)

        instances = outputs['instances'].to('cpu')
        H_f, W_f  = frame.shape[:2]
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        overlay   = frame_rgb.copy()

        if instances.has('pred_densepose') and len(instances) > 0:
            dp          = instances.pred_densepose
            boxes       = instances.pred_boxes.tensor.numpy().astype(int)
            part_labels = dp.fine_segm.argmax(dim=1).numpy()
            fg_masks    = dp.coarse_segm.argmax(dim=1).numpy()  # 1=person, 0=bg

            for lmap, fg, box in zip(part_labels, fg_masks, boxes):
                x1 = max(0, box[0]); y1 = max(0, box[1])
                x2 = min(W_f, box[2]); y2 = min(H_f, box[3])
                if x2 <= x1 or y2 <= y1:
                    continue
                bw, bh = x2 - x1, y2 - y1
                lmap_r = cv2.resize(lmap.astype(np.uint8), (bw, bh), interpolation=cv2.INTER_NEAREST)
                fg_r   = cv2.resize(fg.astype(np.uint8),   (bw, bh), interpolation=cv2.INTER_NEAREST)
                for pid in range(1, 25):
                    mask = (lmap_r == pid) & (fg_r == 1)
                    if mask.any():
                        overlay[y1:y2, x1:x2][mask] = PART_COLORS[pid]

        blended      = (OVERLAY_ALPHA * overlay + (1 - OVERLAY_ALPHA) * frame_rgb).astype(np.uint8)
        last_vis_bgr = cv2.cvtColor(blended, cv2.COLOR_RGB2BGR)

    writer.write(last_vis_bgr if last_vis_bgr is not None else frame)

cap.release()
writer.release()

OUT_VIDEO = OUTPUT_DIR / f'{stem}_densepose.mp4'
subprocess.run([
    'ffmpeg', '-y', '-i', TMP_PATH,
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '20',
    str(OUT_VIDEO)
], check=True, capture_output=True)

size_mb = OUT_VIDEO.stat().st_size / 1e6
print(f'Video saved → {OUT_VIDEO}  ({size_mb:.1f} MB)')